# Tahap 1 - Membangun Case Base

Project: Case-Based Reasoning untuk Pidana Umum - Pencurian di PN Tangerang

Notebook ini digunakan sebagai bagian dari pipeline CBR.


In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

# Project paths
BASE_DIR = Path("..").resolve()

DATA_DIR = BASE_DIR / "data"
RAW_DIR = DATA_DIR / "raw"
PROCESSED_DIR = DATA_DIR / "processed"
EVAL_DIR = DATA_DIR / "eval"
RESULTS_DIR = DATA_DIR / "results"
LOGS_DIR = BASE_DIR / "logs"

# Create folders if not exist
for folder in [RAW_DIR, PROCESSED_DIR, EVAL_DIR, RESULTS_DIR, LOGS_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

print("BASE_DIR:", BASE_DIR)
print("RAW_DIR:", RAW_DIR)
print("PROCESSED_DIR:", PROCESSED_DIR)

BASE_DIR: /Users/bagasfernanda/Desktop/cbr-pencurian-pn-tangerang
RAW_DIR: /Users/bagasfernanda/Desktop/cbr-pencurian-pn-tangerang/data/raw
PROCESSED_DIR: /Users/bagasfernanda/Desktop/cbr-pencurian-pn-tangerang/data/processed


In [2]:
# Membuat template inventory untuk 40 putusan PN Tangerang - Pidana Umum Pencurian

jumlah_putusan = 40

inventory = pd.DataFrame({
    "case_id": [f"case_{i:03d}" for i in range(1, jumlah_putusan + 1)],
    "no_perkara": ["" for _ in range(jumlah_putusan)],
    "pengadilan": ["PN Tangerang" for _ in range(jumlah_putusan)],
    "jenis_perkara": ["Pidana Umum - Pencurian" for _ in range(jumlah_putusan)],
    "tanggal_putusan": ["" for _ in range(jumlah_putusan)],
    "sumber_url": ["" for _ in range(jumlah_putusan)],
    "raw_file": [f"case_{i:03d}.txt" for i in range(1, jumlah_putusan + 1)],
    "status_download": ["belum" for _ in range(jumlah_putusan)],
    "jumlah_kata": [0 for _ in range(jumlah_putusan)]
})

inventory_path = PROCESSED_DIR / "case_inventory.csv"
inventory.to_csv(inventory_path, index=False)

print("Template inventory berhasil dibuat:")
print(inventory_path)

inventory.head()

Template inventory berhasil dibuat:
/Users/bagasfernanda/Desktop/cbr-pencurian-pn-tangerang/data/processed/case_inventory.csv


,case_id,no_perkara,pengadilan,jenis_perkara,tanggal_putusan,sumber_url,raw_file,status_download,jumlah_kata
0,case_001,,PN Tangerang,Pidana Umum - Pencurian,,,case_001.txt,belum,0
1,case_002,,PN Tangerang,Pidana Umum - Pencurian,,,case_002.txt,belum,0
2,case_003,,PN Tangerang,Pidana Umum - Pencurian,,,case_003.txt,belum,0
3,case_004,,PN Tangerang,Pidana Umum - Pencurian,,,case_004.txt,belum,0
4,case_005,,PN Tangerang,Pidana Umum - Pencurian,,,case_005.txt,belum,0


In [6]:
from pathlib import Path
import pandas as pd

BASE_DIR = Path("..").resolve()
inventory_path = BASE_DIR / "data" / "processed" / "case_inventory.csv"

# Baca CSV sebagai teks semua agar tidak error saat isi nomor perkara
df = pd.read_csv(inventory_path, dtype=str).fillna("")

# Pastikan kolom penting ada
required_columns = [
    "case_id",
    "no_perkara",
    "pengadilan",
    "jenis_perkara",
    "tanggal_putusan",
    "sumber_url",
    "raw_file",
    "status_download",
    "jumlah_kata"
]

for col in required_columns:
    if col not in df.columns:
        df[col] = ""

# Semua kolom dibuat string dulu supaya aman
for col in required_columns:
    df[col] = df[col].astype(str)

case_id = "case_006"

# Isi data case_001
df.loc[df["case_id"] == case_id, "no_perkara"] = "310/Pid.B/2019/PN.Tng"
df.loc[df["case_id"] == case_id, "tanggal_putusan"] = "18-03-2019"
df.loc[df["case_id"] == case_id, "pengadilan"] = "PN Tangerang"
df.loc[df["case_id"] == case_id, "jenis_perkara"] = "Pidana Umum - Pencurian"
df.loc[df["case_id"] == case_id, "sumber_url"] = "https://putusan3.mahkamahagung.go.id/direktori/putusan/854b1ce2c8150c85537665183232c221.html"
df.loc[df["case_id"] == case_id, "raw_file"] = "case_006.txt"
df.loc[df["case_id"] == case_id, "status_download"] = "belum"
df.loc[df["case_id"] == case_id, "jumlah_kata"] = "0"

# Simpan ulang
df.to_csv(inventory_path, index=False)

print("Data berhasil diperbarui dan disimpan ke:")
print(inventory_path)

df[df["case_id"] == case_id]

Data berhasil diperbarui dan disimpan ke:
/Users/bagasfernanda/Desktop/cbr-pencurian-pn-tangerang/data/processed/case_inventory.csv


,case_id,no_perkara,pengadilan,jenis_perkara,tanggal_putusan,sumber_url,raw_file,status_download,jumlah_kata
5,case_006,310/Pid.B/2019/PN.Tng,PN Tangerang,Pidana Umum - Pencurian,18-03-2019,https://putusan3.mahkamahagung.go.id/direktori...,case_006.txt,belum,0


In [11]:
rows_text = """
1790 / PID.B / 2014 / PN.TNG.	14 Oktober 2014	https://putusan3.mahkamahagung.go.id/direktori/putusan/7162a09b85f69cff1006bbeb65358019.html
2118/Pid.B/2014/PN.TNG	18 Desember 2014	https://putusan3.mahkamahagung.go.id/direktori/putusan/a90dfad5ed395a96e45243719e858539.html
1996/Pid.B/2011/PN.TNG	22 Desember 2011	https://putusan3.mahkamahagung.go.id/direktori/putusan/7b8578278a899a37e5346fefeec61638.html
265/PID.B/2013/PN.TNG	21 Maret 2013	https://putusan3.mahkamahagung.go.id/direktori/putusan/1952227d2d0bfff610adcb359e6fe52c.html
1380 / PID.B / 2014 / PN.TNG.	28 Agustus 2014	https://putusan3.mahkamahagung.go.id/direktori/putusan/0af7886969da9821c6e5d1c20163fa05.html
1339/ PID.B/ 2011/ PN.TNG.	22 Agustus 2011	https://putusan3.mahkamahagung.go.id/direktori/putusan/33d09168e3bd3a126ab61880e5aa454a.html
1279/Pid.B/2022/PN Tng	25 Oktober 2022	https://putusan3.mahkamahagung.go.id/direktori/putusan/zaed56a5e9aa15deadaa313635303130.html
441/Pid.B/2010/PN.TNG	21 April 2010	https://putusan3.mahkamahagung.go.id/direktori/putusan/1b53a6e54f1d8d2f75b193e9dedcd156.html
917/Pid.B/2019/PN Tng	15 Juli 2019	https://putusan3.mahkamahagung.go.id/direktori/putusan/38755fca77bcfd6d3abbc89817746136.html
1282 / PID.B / 2011 / PN.TNG	10 Agustus 2011	https://putusan3.mahkamahagung.go.id/direktori/putusan/85a681cd7fe2b18180517ffa3351d41e.html
2255/Pid.B/2014/PN.Tng	7 Januari 2015	https://putusan3.mahkamahagung.go.id/direktori/putusan/86bca264b5d325eee8e78dc96460e1a8.html
327/Pid.B/2024/PN Tng	8 Mei 2024	https://putusan3.mahkamahagung.go.id/direktori/putusan/zaef32d1128b9e569b4a313535383231.html
1727/Pid.B/2021/PN Tng	14 Desember 2021	https://putusan3.mahkamahagung.go.id/direktori/putusan/zaec5d838cc3615cbc02313534363532.html
784/Pid.B/2022/PN Tng	20 Juli 2022	https://putusan3.mahkamahagung.go.id/direktori/putusan/zaed14913d95f1ba95dd313433353535.html
1599/PID.B/2010/PN.TNG	22 Nopember 2010	https://putusan3.mahkamahagung.go.id/direktori/putusan/6551f918f664974a8f47356324632764.html
1973/PID.B/2013/PN.TNG	7 Nopember 2013	https://putusan3.mahkamahagung.go.id/direktori/putusan/efec6384c3624a6ba0312fa37c71a975.html
752/Pid.B/2022/PN Tng	20 Juni 2022	https://putusan3.mahkamahagung.go.id/direktori/putusan/zaecf06be94a1ef4b329313433383030.html
110/PID.B/2012/PN.TNG	23 Februari 2012	https://putusan3.mahkamahagung.go.id/direktori/putusan/db413e1ba15e6bf2febeed1aa93f1e95.html
1644/Pid.B/2021/PN Tng	14 Desember 2021	https://putusan3.mahkamahagung.go.id/direktori/putusan/zaec5d83910435d4a3d6313534363539.html
1430/Pid.B/2022/PN Tng	2 Nopember 2022	https://putusan3.mahkamahagung.go.id/direktori/putusan/zaedd2c667f2c0fcbc57313535353130.html
32/Pdt.P/2025/PN Psp		https://putusan3.mahkamahagung.go.id/direktori/putusan/zaf16cab4e3a9c5cb8ef323032343033.html
292/Pid.B/2025/PN Psp		https://putusan3.mahkamahagung.go.id/direktori/putusan/zaf16cab4c54b6fcbc41323032343030.html
272/Pid.B/2025/PN Psp		https://putusan3.mahkamahagung.go.id/direktori/putusan/zaf16cab4a5d78e89838323032333536.html
"""

In [12]:
from pathlib import Path
import pandas as pd

BASE_DIR = Path("..").resolve()
inventory_path = BASE_DIR / "data" / "processed" / "case_inventory.csv"

df = pd.read_csv(inventory_path, dtype=str).fillna("")

required_columns = [
    "case_id",
    "no_perkara",
    "pengadilan",
    "jenis_perkara",
    "tanggal_putusan",
    "sumber_url",
    "raw_file",
    "status_download",
    "jumlah_kata"
]

for col in required_columns:
    if col not in df.columns:
        df[col] = ""

for col in required_columns:
    df[col] = df[col].astype(str)

rows = []

for line in rows_text.strip().splitlines():
    parts = line.strip().split("\t")
    
    if len(parts) >= 3:
        no_perkara = parts[0].strip()
        tanggal_putusan = parts[1].strip()
        url = parts[2].strip()
        
        if url.startswith("http") and "/direktori/putusan/" in url:
            rows.append({
                "no_perkara": no_perkara,
                "tanggal_putusan": tanggal_putusan,
                "sumber_url": url
            })

seen = set()
clean_rows = []

for row in rows:
    if row["sumber_url"] not in seen:
        seen.add(row["sumber_url"])
        clean_rows.append(row)

print(f"Jumlah data valid dari browser: {len(clean_rows)}")

existing_urls = set(df["sumber_url"].dropna().astype(str).tolist())

added = 0

for row in clean_rows:
    url = row["sumber_url"]

    if url in existing_urls:
        print(f"Dilewati karena sudah ada: {url}")
        continue

    empty_rows = df.index[(df["sumber_url"] == "") | (df["sumber_url"].isna())].tolist()

    if not empty_rows:
        print("Tidak ada baris kosong lagi.")
        break

    idx = empty_rows[0]
    case_id = df.loc[idx, "case_id"]

    if case_id == "" or case_id.lower() == "nan":
        case_id = f"case_{idx+1:03d}"
        df.loc[idx, "case_id"] = case_id

    df.loc[idx, "no_perkara"] = row["no_perkara"]
    df.loc[idx, "tanggal_putusan"] = row["tanggal_putusan"]
    df.loc[idx, "pengadilan"] = "PN Tangerang"
    df.loc[idx, "jenis_perkara"] = "Pidana Umum - Pencurian"
    df.loc[idx, "sumber_url"] = url
    df.loc[idx, "raw_file"] = f"{case_id}.txt"
    df.loc[idx, "status_download"] = "belum"
    df.loc[idx, "jumlah_kata"] = "0"

    existing_urls.add(url)
    added += 1

df.to_csv(inventory_path, index=False)

print(f"Berhasil menambahkan {added} data baru.")
print("File disimpan ke:", inventory_path)

df[["case_id", "no_perkara", "tanggal_putusan", "sumber_url", "raw_file", "status_download", "jumlah_kata"]].head(40)

Jumlah data valid dari browser: 23
Tidak ada baris kosong lagi.
Berhasil menambahkan 0 data baru.
File disimpan ke: /Users/bagasfernanda/Desktop/cbr-pencurian-pn-tangerang/data/processed/case_inventory.csv


,case_id,no_perkara,tanggal_putusan,sumber_url,raw_file,status_download,jumlah_kata
0,case_001,2184/PID.B/2013/PN.TNG,07-01-2014,https://putusan3.mahkamahagung.go.id/direktori...,case_001.txt,belum,0
1,case_002,1671/Pid.B/2021/PN.Tng,18-10-2021,https://putusan3.mahkamahagung.go.id/direktori...,case_002.txt,belum,0
2,case_003,1885/Pid.B/2022/PN.Tng,10-10-2022,https://putusan3.mahkamahagung.go.id/direktori...,case_003.txt,belum,0
3,case_004,1314/PID.B/2013/PN.TNG,24-07-2013,https://putusan3.mahkamahagung.go.id/direktori...,case_004.txt,belum,0
4,case_005,1493/PID.B/2013/PN.TNG,27-08-2013,https://putusan3.mahkamahagung.go.id/direktori...,case_005.txt,belum,0
5,case_006,310/Pid.B/2019/PN.Tng,18-03-2019,https://putusan3.mahkamahagung.go.id/direktori...,case_006.txt,belum,0
6,case_007,2686/Pid.B/2018/PN Tng,30 Januari 2019,https://putusan3.mahkamahagung.go.id/direktori...,case_007.txt,belum,0
7,case_008,1581/Pid.B/2011/PN.TNG,4 Oktober 2011,https://putusan3.mahkamahagung.go.id/direktori...,case_008.txt,belum,0
8,case_009,1022/Pid.B/2010/PN.TNG,14 Juli 2010,https://putusan3.mahkamahagung.go.id/direktori...,case_009.txt,belum,0
9,case_010,497 / PID.B / 2014 / PN.TNG.,19 Mei 2014,https://putusan3.mahkamahagung.go.id/direktori...,case_010.txt,belum,0


In [13]:
from pathlib import Path
import pandas as pd

BASE_DIR = Path("..").resolve()
inventory_path = BASE_DIR / "data" / "processed" / "case_inventory.csv"

df = pd.read_csv(inventory_path, dtype=str).fillna("")

jumlah_data = (df["sumber_url"] != "").sum()
jumlah_duplikat = df["sumber_url"].duplicated().sum()

print("Jumlah URL terisi:", jumlah_data)
print("Jumlah URL duplikat:", jumlah_duplikat)

df[["case_id", "no_perkara", "tanggal_putusan", "sumber_url", "raw_file", "status_download", "jumlah_kata"]].head(40)

Jumlah URL terisi: 40
Jumlah URL duplikat: 0


,case_id,no_perkara,tanggal_putusan,sumber_url,raw_file,status_download,jumlah_kata
0,case_001,2184/PID.B/2013/PN.TNG,07-01-2014,https://putusan3.mahkamahagung.go.id/direktori...,case_001.txt,belum,0
1,case_002,1671/Pid.B/2021/PN.Tng,18-10-2021,https://putusan3.mahkamahagung.go.id/direktori...,case_002.txt,belum,0
2,case_003,1885/Pid.B/2022/PN.Tng,10-10-2022,https://putusan3.mahkamahagung.go.id/direktori...,case_003.txt,belum,0
3,case_004,1314/PID.B/2013/PN.TNG,24-07-2013,https://putusan3.mahkamahagung.go.id/direktori...,case_004.txt,belum,0
4,case_005,1493/PID.B/2013/PN.TNG,27-08-2013,https://putusan3.mahkamahagung.go.id/direktori...,case_005.txt,belum,0
5,case_006,310/Pid.B/2019/PN.Tng,18-03-2019,https://putusan3.mahkamahagung.go.id/direktori...,case_006.txt,belum,0
6,case_007,2686/Pid.B/2018/PN Tng,30 Januari 2019,https://putusan3.mahkamahagung.go.id/direktori...,case_007.txt,belum,0
7,case_008,1581/Pid.B/2011/PN.TNG,4 Oktober 2011,https://putusan3.mahkamahagung.go.id/direktori...,case_008.txt,belum,0
8,case_009,1022/Pid.B/2010/PN.TNG,14 Juli 2010,https://putusan3.mahkamahagung.go.id/direktori...,case_009.txt,belum,0
9,case_010,497 / PID.B / 2014 / PN.TNG.,19 Mei 2014,https://putusan3.mahkamahagung.go.id/direktori...,case_010.txt,belum,0


In [14]:
from pathlib import Path
import pandas as pd
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin
import re
import time

BASE_DIR = Path("..").resolve()

RAW_DIR = BASE_DIR / "data" / "raw"
PROCESSED_DIR = BASE_DIR / "data" / "processed"
LOGS_DIR = BASE_DIR / "logs"

RAW_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
LOGS_DIR.mkdir(parents=True, exist_ok=True)

inventory_path = PROCESSED_DIR / "case_inventory.csv"
log_path = LOGS_DIR / "download.log"

df = pd.read_csv(inventory_path, dtype=str).fillna("")

session = requests.Session()
session.headers.update({
    "User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0 Safari/537.36",
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
    "Accept-Language": "id-ID,id;q=0.9,en-US;q=0.8,en;q=0.7",
    "Referer": "https://putusan3.mahkamahagung.go.id/"
})


def clean_text(text):
    text = re.sub(r"\r", "\n", text)
    text = re.sub(r"\t", " ", text)
    text = re.sub(r"\n+", "\n", text)
    text = re.sub(r"[ ]+", " ", text)
    text = text.strip()
    return text


def html_to_text(html):
    soup = BeautifulSoup(html, "html.parser")

    for tag in soup(["script", "style", "nav", "footer", "header"]):
        tag.decompose()

    text = soup.get_text(separator="\n")
    text = clean_text(text)
    return text


def fetch_text_from_url(url):
    response = session.get(url, timeout=30)

    if response.status_code != 200:
        return "", f"gagal_{response.status_code}"

    html = response.text
    text = html_to_text(html)

    if len(text.split()) < 50:
        return text, "berhasil_tapi_teks_pendek"

    return text, "berhasil"


logs = []

for idx, row in df.iterrows():
    case_id = row.get("case_id", f"case_{idx+1:03d}")
    url = row.get("sumber_url", "")
    raw_file = row.get("raw_file", f"{case_id}.txt")

    if not url:
        df.loc[idx, "status_download"] = "url_kosong"
        continue

    output_path = RAW_DIR / raw_file

    print(f"Memproses {case_id}: {url}")

    try:
        text, status = fetch_text_from_url(url)
        jumlah_kata = len(text.split())

        if text:
            output_path.write_text(text, encoding="utf-8")

        df.loc[idx, "status_download"] = status
        df.loc[idx, "jumlah_kata"] = str(jumlah_kata)

        logs.append(f"{case_id} | {status} | {jumlah_kata} kata | {url}")

    except Exception as e:
        df.loc[idx, "status_download"] = "gagal_error"
        df.loc[idx, "jumlah_kata"] = "0"

        logs.append(f"{case_id} | gagal_error | {str(e)} | {url}")

    time.sleep(1)

df.to_csv(inventory_path, index=False)

log_path.write_text("\n".join(logs), encoding="utf-8")

print("\nProses selesai.")
print("Inventory diperbarui:", inventory_path)
print("Log tersimpan:", log_path)

df[["case_id", "status_download", "jumlah_kata", "raw_file"]].head(40)

Memproses case_001: https://putusan3.mahkamahagung.go.id/direktori/putusan/920f583c6a4c4c1e857abd142bcaae4c.html
Memproses case_002: https://putusan3.mahkamahagung.go.id/direktori/putusan/zaec79c6b87c3daa8982313435383134.html
Memproses case_003: https://putusan3.mahkamahagung.go.id/direktori/putusan/zaed970b08fe45feb40e313533353136.html
Memproses case_004: https://putusan3.mahkamahagung.go.id/direktori/putusan/bd7607df513ced610ce107010c47cc1b.html
Memproses case_005: https://putusan3.mahkamahagung.go.id/direktori/putusan/adba62392c732ab4baa77debec9bbd72.html
Memproses case_006: https://putusan3.mahkamahagung.go.id/direktori/putusan/854b1ce2c8150c85537665183232c221.html
Memproses case_007: https://putusan3.mahkamahagung.go.id/direktori/putusan/ae04fe178665082c029a9b8bab0f2e4a.html
Memproses case_008: https://putusan3.mahkamahagung.go.id/direktori/putusan/25a935db4781ac365604657c7a16d549.html
Memproses case_009: https://putusan3.mahkamahagung.go.id/direktori/putusan/02d8373b52a74b64cb2b5

,case_id,status_download,jumlah_kata,raw_file
0,case_001,gagal_403,0,case_001.txt
1,case_002,gagal_403,0,case_002.txt
2,case_003,gagal_403,0,case_003.txt
3,case_004,gagal_403,0,case_004.txt
4,case_005,gagal_403,0,case_005.txt
5,case_006,gagal_403,0,case_006.txt
6,case_007,gagal_403,0,case_007.txt
7,case_008,gagal_403,0,case_008.txt
8,case_009,gagal_403,0,case_009.txt
9,case_010,gagal_403,0,case_010.txt


In [15]:
!pip install selenium

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.5/9.5 MB 2.9 MB/s  0:00:032.9 MB/s eta 0:00:01:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8/8 [selenium]━━ 7/8 [selenium]


In [16]:
from pathlib import Path
import pandas as pd
import time
import re

from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By

BASE_DIR = Path("..").resolve()

RAW_DIR = BASE_DIR / "data" / "raw"
PROCESSED_DIR = BASE_DIR / "data" / "processed"
LOGS_DIR = BASE_DIR / "logs"

RAW_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
LOGS_DIR.mkdir(parents=True, exist_ok=True)

inventory_path = PROCESSED_DIR / "case_inventory.csv"
log_path = LOGS_DIR / "selenium_download.log"

df = pd.read_csv(inventory_path, dtype=str).fillna("")


def clean_text(text):
    text = re.sub(r"\r", "\n", text)
    text = re.sub(r"\t", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    text = re.sub(r"[ ]{2,}", " ", text)
    return text.strip()


def extract_no_perkara_from_text(text):
    patterns = [
        r"Nomor\s*[:\-]?\s*([0-9]+\/Pid\.B\/[0-9]{4}\/PN\.?\s*Tng)",
        r"Nomor\s*[:\-]?\s*([0-9]+\/PID\.B\/[0-9]{4}\/PN\.?\s*TNG)",
        r"([0-9]+\/Pid\.B\/[0-9]{4}\/PN\.?\s*Tng)",
        r"([0-9]+\/PID\.B\/[0-9]{4}\/PN\.?\s*TNG)",
    ]

    for pattern in patterns:
        match = re.search(pattern, text, re.IGNORECASE)
        if match:
            return match.group(1).replace(" ", "")
    
    return ""


def extract_tanggal_from_text(text):
    bulan = "Januari|Februari|Maret|April|Mei|Juni|Juli|Agustus|September|Oktober|November|Desember"

    patterns = [
        rf"tanggal\s+([0-9]{{1,2}}\s+(?:{bulan})\s+[0-9]{{4}})",
        rf"putus\s+[:\-]?\s*([0-9]{{1,2}}\s+(?:{bulan})\s+[0-9]{{4}})",
        rf"([0-9]{{1,2}}\s+(?:{bulan})\s+[0-9]{{4}})",
    ]

    for pattern in patterns:
        match = re.search(pattern, text, re.IGNORECASE)
        if match:
            return match.group(1)
    
    return ""


# Setting Chrome
chrome_options = Options()

# Jangan headless dulu supaya seperti browser biasa
# Chrome akan terbuka otomatis
chrome_options.add_argument("--start-maximized")

driver = webdriver.Chrome(options=chrome_options)

logs = []

try:
    for idx, row in df.iterrows():
        case_id = row.get("case_id", f"case_{idx+1:03d}")
        url = row.get("sumber_url", "")
        raw_file = row.get("raw_file", f"{case_id}.txt")

        if not url:
            df.loc[idx, "status_download"] = "url_kosong"
            df.loc[idx, "jumlah_kata"] = "0"
            continue

        output_path = RAW_DIR / raw_file

        print(f"\nMemproses {case_id}")
        print(url)

        try:
            driver.get(url)

            # beri waktu halaman terbuka
            time.sleep(4)

            body = driver.find_element(By.TAG_NAME, "body")
            text = body.text
            text = clean_text(text)

            jumlah_kata = len(text.split())

            if "403" in text[:300] or "forbidden" in text[:300].lower():
                status = "gagal_browser_403"
                jumlah_kata = 0
            elif jumlah_kata < 50:
                status = "berhasil_tapi_teks_pendek"
            else:
                status = "berhasil"

            if jumlah_kata > 0:
                output_path.write_text(text, encoding="utf-8")

            # Update metadata jika masih kosong
            if df.loc[idx, "no_perkara"] == "":
                no_perkara = extract_no_perkara_from_text(text)
                df.loc[idx, "no_perkara"] = no_perkara

            if df.loc[idx, "tanggal_putusan"] == "":
                tanggal_putusan = extract_tanggal_from_text(text)
                df.loc[idx, "tanggal_putusan"] = tanggal_putusan

            df.loc[idx, "status_download"] = status
            df.loc[idx, "jumlah_kata"] = str(jumlah_kata)
            df.loc[idx, "raw_file"] = raw_file

            logs.append(f"{case_id} | {status} | {jumlah_kata} kata | {url}")

            print(f"Status: {status}")
            print(f"Jumlah kata: {jumlah_kata}")

            # Simpan progress setiap 1 kasus agar aman
            df.to_csv(inventory_path, index=False)
            log_path.write_text("\n".join(logs), encoding="utf-8")

        except Exception as e:
            df.loc[idx, "status_download"] = "gagal_error"
            df.loc[idx, "jumlah_kata"] = "0"

            logs.append(f"{case_id} | gagal_error | {str(e)} | {url}")

            print("Gagal:", e)

            df.to_csv(inventory_path, index=False)
            log_path.write_text("\n".join(logs), encoding="utf-8")

        # jeda agar tidak terlalu cepat
        time.sleep(3)

finally:
    driver.quit()

df.to_csv(inventory_path, index=False)
log_path.write_text("\n".join(logs), encoding="utf-8")

print("\nProses Selenium selesai.")
print("Inventory diperbarui:", inventory_path)
print("Log tersimpan:", log_path)

df[["case_id", "no_perkara", "tanggal_putusan", "status_download", "jumlah_kata", "raw_file"]].head(40)


Memproses case_001
https://putusan3.mahkamahagung.go.id/direktori/putusan/920f583c6a4c4c1e857abd142bcaae4c.html
Status: berhasil_tapi_teks_pendek
Jumlah kata: 33

Memproses case_002
https://putusan3.mahkamahagung.go.id/direktori/putusan/zaec79c6b87c3daa8982313435383134.html
Status: berhasil_tapi_teks_pendek
Jumlah kata: 41

Memproses case_003
https://putusan3.mahkamahagung.go.id/direktori/putusan/zaed970b08fe45feb40e313533353136.html
Status: berhasil_tapi_teks_pendek
Jumlah kata: 33

Memproses case_004
https://putusan3.mahkamahagung.go.id/direktori/putusan/bd7607df513ced610ce107010c47cc1b.html
Status: berhasil_tapi_teks_pendek
Jumlah kata: 41

Memproses case_005
https://putusan3.mahkamahagung.go.id/direktori/putusan/adba62392c732ab4baa77debec9bbd72.html
Status: berhasil_tapi_teks_pendek
Jumlah kata: 41

Memproses case_006
https://putusan3.mahkamahagung.go.id/direktori/putusan/854b1ce2c8150c85537665183232c221.html
Status: berhasil_tapi_teks_pendek
Jumlah kata: 41

Memproses case_007
ht

,case_id,no_perkara,tanggal_putusan,status_download,jumlah_kata,raw_file
0,case_001,2184/PID.B/2013/PN.TNG,07-01-2014,berhasil_tapi_teks_pendek,33,case_001.txt
1,case_002,1671/Pid.B/2021/PN.Tng,18-10-2021,berhasil_tapi_teks_pendek,41,case_002.txt
2,case_003,1885/Pid.B/2022/PN.Tng,10-10-2022,berhasil_tapi_teks_pendek,33,case_003.txt
3,case_004,1314/PID.B/2013/PN.TNG,24-07-2013,berhasil_tapi_teks_pendek,41,case_004.txt
4,case_005,1493/PID.B/2013/PN.TNG,27-08-2013,berhasil_tapi_teks_pendek,41,case_005.txt
5,case_006,310/Pid.B/2019/PN.Tng,18-03-2019,berhasil_tapi_teks_pendek,41,case_006.txt
6,case_007,2686/Pid.B/2018/PN Tng,30 Januari 2019,berhasil_tapi_teks_pendek,41,case_007.txt
7,case_008,1581/Pid.B/2011/PN.TNG,4 Oktober 2011,berhasil_tapi_teks_pendek,41,case_008.txt
8,case_009,1022/Pid.B/2010/PN.TNG,14 Juli 2010,berhasil_tapi_teks_pendek,41,case_009.txt
9,case_010,497 / PID.B / 2014 / PN.TNG.,19 Mei 2014,berhasil_tapi_teks_pendek,41,case_010.txt


In [17]:
df = pd.read_csv(inventory_path, dtype=str).fillna("")

print(df["status_download"].value_counts())

df[["case_id", "no_perkara", "tanggal_putusan", "status_download", "jumlah_kata", "raw_file"]].head(40)

status_download
berhasil_tapi_teks_pendek    40
Name: count, dtype: int64


,case_id,no_perkara,tanggal_putusan,status_download,jumlah_kata,raw_file
0,case_001,2184/PID.B/2013/PN.TNG,07-01-2014,berhasil_tapi_teks_pendek,33,case_001.txt
1,case_002,1671/Pid.B/2021/PN.Tng,18-10-2021,berhasil_tapi_teks_pendek,41,case_002.txt
2,case_003,1885/Pid.B/2022/PN.Tng,10-10-2022,berhasil_tapi_teks_pendek,33,case_003.txt
3,case_004,1314/PID.B/2013/PN.TNG,24-07-2013,berhasil_tapi_teks_pendek,41,case_004.txt
4,case_005,1493/PID.B/2013/PN.TNG,27-08-2013,berhasil_tapi_teks_pendek,41,case_005.txt
5,case_006,310/Pid.B/2019/PN.Tng,18-03-2019,berhasil_tapi_teks_pendek,41,case_006.txt
6,case_007,2686/Pid.B/2018/PN Tng,30 Januari 2019,berhasil_tapi_teks_pendek,41,case_007.txt
7,case_008,1581/Pid.B/2011/PN.TNG,4 Oktober 2011,berhasil_tapi_teks_pendek,41,case_008.txt
8,case_009,1022/Pid.B/2010/PN.TNG,14 Juli 2010,berhasil_tapi_teks_pendek,41,case_009.txt
9,case_010,497 / PID.B / 2014 / PN.TNG.,19 Mei 2014,berhasil_tapi_teks_pendek,41,case_010.txt


In [18]:
from pathlib import Path
import pandas as pd

BASE_DIR = Path("..").resolve()
RAW_DIR = BASE_DIR / "data" / "raw"
PROCESSED_DIR = BASE_DIR / "data" / "processed"

inventory_path = PROCESSED_DIR / "case_inventory.csv"

df = pd.read_csv(inventory_path, dtype=str).fillna("")

def is_cloudflare_text(text):
    markers = [
        "cloudflare",
        "melakukan verifikasi keamanan",
        "ray id",
        "bot jahat",
        "performa dan keamanan",
        "privasi"
    ]
    text_lower = text.lower()
    return any(marker in text_lower for marker in markers)

reset_count = 0

for idx, row in df.iterrows():
    raw_file = row.get("raw_file", "")
    raw_path = RAW_DIR / raw_file

    if raw_path.exists():
        text = raw_path.read_text(encoding="utf-8", errors="ignore")
        
        if is_cloudflare_text(text):
            raw_path.unlink()
            df.loc[idx, "status_download"] = "gagal_cloudflare"
            df.loc[idx, "jumlah_kata"] = "0"
            reset_count += 1

df.to_csv(inventory_path, index=False)

print("Jumlah file Cloudflare yang dihapus/reset:", reset_count)

df[["case_id", "status_download", "jumlah_kata", "raw_file"]].head(40)

Jumlah file Cloudflare yang dihapus/reset: 40


,case_id,status_download,jumlah_kata,raw_file
0,case_001,gagal_cloudflare,0,case_001.txt
1,case_002,gagal_cloudflare,0,case_002.txt
2,case_003,gagal_cloudflare,0,case_003.txt
3,case_004,gagal_cloudflare,0,case_004.txt
4,case_005,gagal_cloudflare,0,case_005.txt
5,case_006,gagal_cloudflare,0,case_006.txt
6,case_007,gagal_cloudflare,0,case_007.txt
7,case_008,gagal_cloudflare,0,case_008.txt
8,case_009,gagal_cloudflare,0,case_009.txt
9,case_010,gagal_cloudflare,0,case_010.txt


In [19]:
from pathlib import Path
import pandas as pd
import time
import re

from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By

BASE_DIR = Path("..").resolve()

RAW_DIR = BASE_DIR / "data" / "raw"
PROCESSED_DIR = BASE_DIR / "data" / "processed"
LOGS_DIR = BASE_DIR / "logs"
CHROME_PROFILE_DIR = BASE_DIR / "chrome_profile"

RAW_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
LOGS_DIR.mkdir(parents=True, exist_ok=True)
CHROME_PROFILE_DIR.mkdir(parents=True, exist_ok=True)

inventory_path = PROCESSED_DIR / "case_inventory.csv"
log_path = LOGS_DIR / "selenium_download_retry.log"

df = pd.read_csv(inventory_path, dtype=str).fillna("")


def clean_text(text):
    text = re.sub(r"\r", "\n", text)
    text = re.sub(r"\t", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    text = re.sub(r"[ ]{2,}", " ", text)
    return text.strip()


def is_security_page(text):
    markers = [
        "cloudflare",
        "melakukan verifikasi keamanan",
        "ray id",
        "bot jahat",
        "performa dan keamanan",
        "privacy",
        "privasi"
    ]
    text_lower = text.lower()
    return any(marker in text_lower for marker in markers)


def extract_page_text(driver):
    body = driver.find_element(By.TAG_NAME, "body")
    return clean_text(body.text)


chrome_options = Options()
chrome_options.add_argument("--start-maximized")

# Profile khusus agar cookie/verifikasi tersimpan
chrome_options.add_argument(f"--user-data-dir={CHROME_PROFILE_DIR}")

driver = webdriver.Chrome(options=chrome_options)

logs = []

try:
    # Buka URL pertama dulu untuk verifikasi awal
    first_url = df[df["sumber_url"] != ""].iloc[0]["sumber_url"]
    driver.get(first_url)

    print("Chrome sudah terbuka.")
    print("Kalau muncul halaman verifikasi Cloudflare, tunggu sampai halaman putusan benar-benar tampil.")
    print("Setelah halaman putusan tampil normal, tekan Enter di Jupyter ini.")
    input("Tekan Enter setelah verifikasi selesai dan halaman putusan terbuka... ")

    for idx, row in df.iterrows():
        case_id = row.get("case_id", f"case_{idx+1:03d}")
        url = row.get("sumber_url", "")
        raw_file = row.get("raw_file", f"{case_id}.txt")
        output_path = RAW_DIR / raw_file

        if not url:
            df.loc[idx, "status_download"] = "url_kosong"
            df.loc[idx, "jumlah_kata"] = "0"
            continue

        # Lewati jika sudah berhasil dan file ada
        if output_path.exists() and row.get("status_download", "") == "berhasil":
            continue

        print(f"\nMemproses {case_id}")
        print(url)

        try:
            driver.get(url)

            text = ""

            # Tunggu maksimal 30 detik sampai bukan halaman security
            for second in range(30):
                time.sleep(1)
                text = extract_page_text(driver)

                if not is_security_page(text) and len(text.split()) > 80:
                    break

            text = clean_text(text)
            jumlah_kata = len(text.split())

            if is_security_page(text):
                status = "gagal_cloudflare"
                jumlah_kata = 0
                print("Masih halaman Cloudflare.")
            elif jumlah_kata < 80:
                status = "berhasil_tapi_teks_pendek"
                output_path.write_text(text, encoding="utf-8")
                print("Teks terlalu pendek.")
            else:
                status = "berhasil"
                output_path.write_text(text, encoding="utf-8")
                print("Berhasil ambil teks.")

            df.loc[idx, "status_download"] = status
            df.loc[idx, "jumlah_kata"] = str(jumlah_kata)
            df.loc[idx, "raw_file"] = raw_file

            logs.append(f"{case_id} | {status} | {jumlah_kata} kata | {url}")

            print("Status:", status)
            print("Jumlah kata:", jumlah_kata)

            df.to_csv(inventory_path, index=False)
            log_path.write_text("\n".join(logs), encoding="utf-8")

        except Exception as e:
            df.loc[idx, "status_download"] = "gagal_error"
            df.loc[idx, "jumlah_kata"] = "0"

            logs.append(f"{case_id} | gagal_error | {str(e)} | {url}")

            print("Gagal:", e)

            df.to_csv(inventory_path, index=False)
            log_path.write_text("\n".join(logs), encoding="utf-8")

        time.sleep(3)

finally:
    driver.quit()

df.to_csv(inventory_path, index=False)
log_path.write_text("\n".join(logs), encoding="utf-8")

print("\nRetry selesai.")
print("Inventory diperbarui:", inventory_path)

df[["case_id", "status_download", "jumlah_kata", "raw_file"]].head(40)

Chrome sudah terbuka.
Kalau muncul halaman verifikasi Cloudflare, tunggu sampai halaman putusan benar-benar tampil.
Setelah halaman putusan tampil normal, tekan Enter di Jupyter ini.


Tekan Enter setelah verifikasi selesai dan halaman putusan terbuka...  



Memproses case_001
https://putusan3.mahkamahagung.go.id/direktori/putusan/920f583c6a4c4c1e857abd142bcaae4c.html


KeyboardInterrupt: 